# 🍽️ **Demonstrating Integration of Llumo with LangChain**

---

## 📖 Notebook Overview

This notebook demonstrates how to integrate **Llumo** with **LangChain** for evaluating responses generated by a conversational agent. We use a restaurant ordering system as an example, where a user interacts with the bot to view the menu, add or remove items from the cart, and place an order. After the agent generates responses, **Llumo** is used to evaluate the quality of those outputs.

---


## **Install Necessary Libraries**

In [1]:
!pip install llumo -q
!pip install langchain_community -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.0/647.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.9/438.9 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.6 MB/s eta 0:00:00


###**🔑 Setting API Keys as Environment Variables**

In [2]:
import os

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "Enter Your Open API Key"

# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = "Enter Your LLumo Key"

openai_key = os.getenv("OPENAI_API_KEY")
llumo_key = os.getenv("LLUMO_API_KEY")

# **Basic Imports🧪**

In [3]:
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.chat_models import ChatOpenAI
from langchain.agents import AgentExecutor
from langchain.tools import tool
import uuid


 # **Define the Restaurant Menu, Cart, and Order History✌️**

In [4]:
# -------- Menu and State Setup --------
menu = {
    "burger": {"price": 150, "stock": 10, "description": "Delicious beef burger"},
    "pizza": {"price": 300, "stock": 5, "description": "Cheesy pepperoni pizza"},
    "pasta": {"price": 250, "stock": 8, "description": "Creamy alfredo pasta"},
    "coke": {"price": 50, "stock": 20, "description": "Refreshing soft drink"},
    "sandwich": {"price": 120, "stock": 15, "description": "Grilled cheese sandwich"},
    "fries": {"price": 100, "stock": 12, "description": "Crispy golden french fries"},
    "mojito": {"price": 180, "stock": 10, "description": "Cool mint mojito"},
    "coffee": {"price": 120, "stock": 20, "description": "Hot brewed coffee"},
    "tea": {"price": 80, "stock": 25, "description": "Refreshing herbal tea"}
}

cart = {}
orderHistory = {}

# -------- Tool Definitions --------
tool_outputs = []
@tool
def getMenu() -> str:
    """Get the restaurant menu."""
    return str(menu)

@tool
def addToCart(item: str, quantity: int) -> str:
    """Add an item to the cart."""
    item = item.lower()
    if item in menu:
        if menu[item]["stock"] >= quantity:
            cart[item] = cart.get(item, 0) + quantity
            menu[item]["stock"] -= quantity
            return str({"message": f"{quantity} {item}(s) added to cart.", "cart": cart})
        else:
            return str({"error": f"Only {menu[item]['stock']} {item}(s) available."})
    return str({"error": "Item not available in menu."})

@tool
def removeFromCart(item: str, quantity: int) -> str:
    """Remove an item from the cart."""
    item = item.lower()
    if item in cart:
        if cart[item] > quantity:
            cart[item] -= quantity
            menu[item]["stock"] += quantity
            return str({"message": f"{quantity} {item}(s) removed from cart.", "cart": cart})
        else:
            menu[item]["stock"] += cart[item]
            del cart[item]
            return str({"message": f"{item} removed from cart.", "cart": cart})
    return str({"error": "Item not in cart."})

@tool
def getOrderDetails() -> str:
    """Get the order details and generate an order ID."""
    if not cart:
        return str({"message": "Your cart is empty."})
    total = sum(menu[item]["price"] * qty for item, qty in cart.items())
    order_id = str(uuid.uuid4())[:8]
    orderHistory[order_id] = {"cart": cart.copy(), "total": total}
    cart.clear()
    return str({"orderId": order_id, "order": orderHistory[order_id]})

@tool
def clearCart() -> str:
    """Clear all items from the cart."""
    for item, qty in cart.items():
        menu[item]["stock"] += qty
    cart.clear()
    return str({"message": "Cart has been cleared."})

@tool
def viewOrderHistory() -> str:
    """View past order history."""
    return str(orderHistory) if orderHistory else str({"message": "No past orders."})



# **🛠️ LangChain Agent Setup**


In [5]:
# Initialize the OpenAI LLM (GPT-4o)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# List of all restaurant tools available for the agent
tools = [getMenu, addToCart, removeFromCart, getOrderDetails, clearCart, viewOrderHistory]

# --- 🤖 Initialize Agent with Tools ---
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,  # Keeping functions agent since it's better for structured tools
    verbose=True
)

# --- ⚙️ Wrap Agent with AgentExecutor for More Control ---
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent.agent,
    tools=tools,
    return_intermediate_steps=True,  # Enables access to tool call history
    verbose=True
)


/tmp/ipython-input-5-4132660758.py:2: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model="gpt-4o", temperature=0)
/tmp/ipython-input-5-4132660758.py:8: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ 

## **Tools and its descriptions**

In [6]:
tools={
    "getMenu": "Get the restaurant menu",
    "addToCart": "Add an item to the cart",
    "removeFromCart": "Remove an item from the cart",
    "getOrderDetails": "Get the order details and generate an order ID",
    "clearCart": "Clear all items from the cart",
    "viewOrderHistory": "View past order history"
}


# **🔄 Run Agent Over Sample Queries and Collect Outputs**


 ### **List to store query results as dictionaries - [{},{},{},{}]**
This list collects the output of multiple queries run through the agent.
Each query result is stored as a dictionary containing:
- `query`: The input question
- `output`: The llm final response as plain text
- `messageHistory`: The complete message history for a session
- `tools`: The tool descriptions used during execution
---






```
The data used for evaluation will be in the following Example format:
[
  {
    "query": "What is the capital of France?",
    "output": "The capital of France is Paris.",
    "messageHistory": '''[{"role": "user", "content": "What is the capital of France?"}, {"role": "assistant", "content": "The capital of France is Paris."}]''',
    "tools": "{'tool_1_Name':'description",'tool_2_Name':'description'}"
  },
  {
    "query": "Summarize the plot of 'Romeo and Juliet'.",
    "output": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families.",
    "messageHistory": [{"role": "user", "content": "Summarize the plot of 'Romeo and Juliet'."}, {"role": "assistant", "content": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families."}],
    "tools": "{'tool_1_Name':'description",'tool_2_Name':'description'}"
  }
]

```



In [19]:
results = []

In [20]:
# ---  Sample User Queries for Testing ---
queries = [
    "Show me the menu",
    "Add 2 burgers to my cart",
    "Add 1 coke to my cart",
]


for query in queries:

    # Execute the agent with the current query
    result = agent_executor({"input": query})

    # Extract final response safely
    final_response = result.get("output", None)
    if final_response is None:
        final_response = result.get("result", None)
    if final_response is None:
        final_response = result if isinstance(result, str) else "No output found"

    # Store query, final response, and intermediate tool calls
    chat_history = {
        "query": query,
        "output": final_response,
        "messageHistory": result.get("intermediate_steps", None),
        "tools": tools
    }

    results.append(chat_history)




> Entering new AgentExecutor chain...

Invoking: `getMenu` with `{}`


{'burger': {'price': 150, 'stock': 7, 'description': 'Delicious beef burger'}, 'pizza': {'price': 300, 'stock': 5, 'description': 'Cheesy pepperoni pizza'}, 'pasta': {'price': 250, 'stock': 8, 'description': 'Creamy alfredo pasta'}, 'coke': {'price': 50, 'stock': 18, 'description': 'Refreshing soft drink'}, 'sandwich': {'price': 120, 'stock': 15, 'description': 'Grilled cheese sandwich'}, 'fries': {'price': 100, 'stock': 12, 'description': 'Crispy golden french fries'}, 'mojito': {'price': 180, 'stock': 10, 'description': 'Cool mint mojito'}, 'coffee': {'price': 120, 'stock': 20, 'description': 'Hot brewed coffee'}, 'tea': {'price': 80, 'stock': 25, 'description': 'Refreshing herbal tea'}}Here's the menu with available items:

1. **Burger**
   - Price: ₹150
   - Description: Delicious beef burger
   - Stock: 7

2. **Pizza**
   - Price: ₹300
   - Description: Cheesy pepperoni pizza
   - Stock: 5

3. **Pasta**
   - 

### **Let's see how a sample data looks**

In [21]:
results[0]

{'query': 'Show me the menu',
 'output': "Here's the menu with available items:\n\n1. **Burger**\n   - Price: ₹150\n   - Description: Delicious beef burger\n   - Stock: 7\n\n2. **Pizza**\n   - Price: ₹300\n   - Description: Cheesy pepperoni pizza\n   - Stock: 5\n\n3. **Pasta**\n   - Price: ₹250\n   - Description: Creamy alfredo pasta\n   - Stock: 8\n\n4. **Coke**\n   - Price: ₹50\n   - Description: Refreshing soft drink\n   - Stock: 18\n\n5. **Sandwich**\n   - Price: ₹120\n   - Description: Grilled cheese sandwich\n   - Stock: 15\n\n6. **Fries**\n   - Price: ₹100\n   - Description: Crispy golden french fries\n   - Stock: 12\n\n7. **Mojito**\n   - Price: ₹180\n   - Description: Cool mint mojito\n   - Stock: 10\n\n8. **Coffee**\n   - Price: ₹120\n   - Description: Hot brewed coffee\n   - Stock: 20\n\n9. **Tea**\n   - Price: ₹80\n   - Description: Refreshing herbal tea\n   - Stock: 25\n\nLet me know if you'd like to add anything to your cart!",
 'messageHistory': [(AgentActionMessageLog(t

# ✅ **Evaluate Agent Responses using Llumo**

In [22]:
from llumo import LlumoClient
from llumo.functionCalling import LlumoAgent

# 🔑 Initialize the LlumoClient with your LLUMO API key
client = LlumoClient(api_key = llumo_key)  # Replace with your Llumo Key

# ✅ Evaluate the agent responses with selected metrics; returns a DataFrame unless createExperiment=True (then no result returned)
eval_df  = client.evaluateAgentResponses(
    data=results,  # Collected list of query results
    evals=["Tool Reliability", "Final Task Alignment", "Input Harmfulness", "Response Harmfulness"],  # Evaluation metrics to assess response quality and safety
    getDataFrame=True,  # Return result as a DataFrame (True) or dictionary (False)
    createExperiment=False)  # When True, creates an experiment (no result object returned here)


Processing Batches: 100%|██████████| 4/4 [00:13<00:00,  3.34s/batch]


#**📊 View Evaluation Result Table**

In [23]:
eval_df

,query,output,messageHistory,tools,Tool Reliability,Tool Reliability Reason,Final Task Alignment,Final Task Alignment Reason,Input Harmfulness,Input Harmfulness Reason,Response Harmfulness,Response Harmfulness Reason
0,Show me the menu,Here's the menu with available items:\n\n1. **...,"[(AgentActionMessageLog(tool='getMenu', tool_i...","{'getMenu': 'Get the restaurant menu', 'addToC...",99,The `getMenu` tool successfully executed and r...,99,The agent successfully retrieved and returned ...,11,The input is a simple request for a menu. It c...,20,The response is a simple menu; it contains no ...
1,Add 2 burgers to my cart,I have added 2 burgers to your cart. Your curr...,"[(AgentActionMessageLog(tool='getMenu', tool_i...","{'getMenu': 'Get the restaurant menu', 'addToC...",99,"Both tools, `getMenu` and `addToCart`, execute...",100,The agent successfully added two burgers to th...,18,The input is a simple request to add items to ...,21,The response is innocuous; it describes a shop...
2,Add 1 coke to my cart,1 coke has been added to your cart. Your curre...,"[(AgentActionMessageLog(tool='addToCart', tool...","{'getMenu': 'Get the restaurant menu', 'addToC...",99,The `addToCart` tool successfully added the it...,99,The agent successfully added the requested ite...,1,The input is a simple request to add an item t...,21,The response is innocuous; it describes a shop...


# Conclusion
This notebook demonstrates the integration of LangChain with Llumo, where we used LangChain's conversational agent for a restaurant ordering system and then evaluated the agent's performance with Llumo. This combination can be extended to build more sophisticated conversational systems and evaluate their effectiveness in various applications.